# Data Cleaning

En este notebook se realiza la exploración y limpieza inicial de los datos utilizando Pandas y Apache Spark.

El objetivo es preparar la información para su posterior uso en modelos de Machine Learning.

In [110]:
# Importar librerías

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.ml.feature import Imputer

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [111]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin")
    .getOrCreate()
)

## Cargar los conjuntos de datos con Pandas

In [112]:
# Cargar datasets

cards_df = pd.read_csv("../data/modified/cards_data.csv")

transactions_df = pd.read_csv("../data/modified/transactions_data.csv")

users_df = pd.read_csv("../data/modified/users_data.csv")

In [113]:
# Mostrar primeras filas

cards_df.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524.0,825.0,Visa,Debit,4.344680e+15,dic-22,623.0,YES,2.0,"$24,295",sep-02,2008.0,No
1,2731.0,825.0,Visa,Debit,4.956970e+15,dic-20,393.0,YES,2.0,"$21,968",abr-14,2014.0,No
2,3701.0,825.0,Visa,Debit,4.582310e+15,feb-24,719.0,YES,2.0,"$46,414",jul-03,2004.0,No
3,42.0,825.0,Visa,Credit,4.879490e+15,ago-24,693.0,NO,1.0,"$12,400",ene-03,2012.0,No
4,4659.0,825.0,Mastercard,Debit (Prepaid),5.722870e+15,mar-09,75.0,YES,1.0,$28,sep-08,2009.0,No


In [114]:
transactions_df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,01/01/2010 00:01,1556.0,2972.0,-$77.00,Swipe Transaction,59935.0,Beulah,ND,58523,5499.0,NaN
1,7475328,01/01/2010 00:02,561.0,4575.0,$14.57,Swipe Transaction,67570.0,Bettendorf,IA,52722,5311.0,NaN
2,7475329,01/01/2010 00:02,1129.0,102.0,$80.00,Swipe Transaction,27092.0,Vista,CA,92084,4829.0,NaN
3,7475331,01/01/2010 00:05,430.0,2860.0,$200.00,Swipe Transaction,27092.0,Crown Point,IN,46307,4829.0,NaN
4,7475332,01/01/2010 00:06,848.0,3915.0,$46.41,Swipe Transaction,13051.0,Harwood,MD,20776,5813.0,NaN


In [115]:
users_df.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53.0,66.0,1966.0,11.0,Female,462 Rose Lane,34.15,-117.76,"$29,278","$59,696","$127,613",787.0,5.0
1,1746,53.0,68.0,1966.0,12.0,Female,3606 Federal Boulevard,40.76,-73.74,"$37,891","$77,254","$191,349",701.0,5.0
2,1718,81.0,67.0,1938.0,11.0,Female,766 Third Drive,34.02,-117.89,"$22,681","$33,483",$196,698.0,5.0
3,708,63.0,63.0,1957.0,1.0,Female,3 Madison Street,40.71,-73.99,"$163,145","$249,925","$202,328",722.0,4.0
4,1164,43.0,70.0,1976.0,9.0,Male,9620 Valley Stream Drive,37.76,-122.44,"$53,797","$109,687","$183,855",675.0,1.0


## Cargar los conjuntos de datos con Apache Spark

In [116]:
cards_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/cards_data.csv")
)

transactions_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/transactions_data.csv")
)

users_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/users_data.csv")
)

In [117]:
cards_spark.show(5)

+----+---------+----------+---------------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  id|client_id|card_brand|      card_type|card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+----+---------+----------+---------------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|4524|      825|      Visa|          Debit| 4.34468E15| dic-22|623|     YES|               2|     $24,295|        sep-02|                 2008|              No|
|2731|      825|      Visa|          Debit| 4.95697E15| dic-20|393|     YES|               2|     $21,968|        abr-14|                 2014|              No|
|3701|      825|      Visa|          Debit| 4.58231E15| feb-24|719|     YES|               2|     $46,414|        jul-03|                 2004|              No|
|  42|      825|      Visa|       

In [118]:
transactions_spark.show(5)

+-------+----------------+---------+-------+-------+-----------------+-----------+-------------+--------------+-----+----+------+
|     id|            date|client_id|card_id| amount|         use_chip|merchant_id|merchant_city|merchant_state|  zip| mcc|errors|
+-------+----------------+---------+-------+-------+-----------------+-----------+-------------+--------------+-----+----+------+
|7475327|01/01/2010 00:01|     1556|   2972|-$77.00|Swipe Transaction|      59935|       Beulah|            ND|58523|5499|  NULL|
|7475328|01/01/2010 00:02|      561|   4575| $14.57|Swipe Transaction|      67570|   Bettendorf|            IA|52722|5311|  NULL|
|7475329|01/01/2010 00:02|     1129|    102| $80.00|Swipe Transaction|      27092|        Vista|            CA|92084|4829|  NULL|
|7475331|01/01/2010 00:05|      430|   2860|$200.00|Swipe Transaction|      27092|  Crown Point|            IN|46307|4829|  NULL|
|7475332|01/01/2010 00:06|      848|   3915| $46.41|Swipe Transaction|      13051|      Ha

In [119]:
users_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|Female|       462 Rose Lane|   34.15|  -117.76|          $29,278|      $59,696|  $127,613|         787|               5|
|1746|         53|            68|      1966|         12|Female|3606 Federal Boul...|   40.76|   -73.74|          $37,891|      $77,254|  $191,349|         701|               5|
|1718|         81|            67|      1938|         11|Female|     766 Third Drive|   34.02|  -117.89|          $2

## Explorar la información

In [120]:
cards_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6746 entries, 0 to 6745
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     6745 non-null   float64
 1   client_id              6745 non-null   float64
 2   card_brand             6674 non-null   str    
 3   card_type              6689 non-null   str    
 4   card_number            6683 non-null   float64
 5   expires                6680 non-null   str    
 6   cvv                    6678 non-null   float64
 7   has_chip               6678 non-null   str    
 8   num_cards_issued       6684 non-null   float64
 9   credit_limit           6655 non-null   str    
 10  acct_open_date         6691 non-null   str    
 11  year_pin_last_changed  6674 non-null   float64
 12  card_on_dark_web       6670 non-null   str    
dtypes: float64(6), str(7)
memory usage: 930.8 KB


In [121]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50598 entries, 0 to 50597
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              50598 non-null  int64  
 1   date            50499 non-null  str    
 2   client_id       50596 non-null  float64
 3   card_id         50596 non-null  float64
 4   amount          50499 non-null  str    
 5   use_chip        50517 non-null  str    
 6   merchant_id     50595 non-null  float64
 7   merchant_city   50499 non-null  str    
 8   merchant_state  45039 non-null  str    
 9   zip             44856 non-null  str    
 10  mcc             50498 non-null  float64
 11  errors          1298 non-null   str    
dtypes: float64(4), int64(1), str(7)
memory usage: 7.3 MB


In [122]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2600 non-null   int64  
 1   current_age        2537 non-null   float64
 2   retirement_age     2535 non-null   float64
 3   birth_year         2537 non-null   float64
 4   birth_month        2545 non-null   float64
 5   gender             2539 non-null   str    
 6   address            2532 non-null   str    
 7   latitude           2550 non-null   float64
 8   longitude          2546 non-null   float64
 9   per_capita_income  2536 non-null   str    
 10  yearly_income      2554 non-null   str    
 11  total_debt         2537 non-null   str    
 12  credit_score       2551 non-null   float64
 13  num_credit_cards   2538 non-null   float64
dtypes: float64(8), int64(1), str(5)
memory usage: 394.0 KB


In [123]:
cards_df.describe()

,id,client_id,card_number,cvv,num_cards_issued,year_pin_last_changed
count,6745.000000,6745.000000,6.683000e+03,6678.000000,6684.000000,6674.000000
mean,3113.004003,1007.760415,4.822923e+15,507.306229,1.503441,2013.457597
std,1824.287261,627.623432,1.327993e+15,290.022485,0.518824,4.289126
min,0.000000,0.000000,3.001060e+14,0.000000,1.000000,2002.000000
25%,1559.000000,494.000000,4.488915e+15,258.000000,1.000000,2010.000000
50%,3095.000000,994.000000,5.112840e+15,516.500000,1.000000,2013.000000
75%,4642.000000,1502.000000,5.587045e+15,759.000000,2.000000,2017.000000
max,11108.000000,6954.000000,6.997200e+15,999.000000,3.000000,2020.000000


In [124]:
transactions_df.describe()

,id,client_id,card_id,merchant_id,mcc
count,5.059800e+04,50596.000000,50596.000000,50595.000000,50498.000000
mean,7.505268e+06,1034.678611,3430.538422,47627.852772,5559.147531
std,1.728141e+04,595.658814,1704.994773,25836.751133,861.981474
min,7.475327e+06,0.000000,0.000000,22.000000,1711.000000
25%,7.490325e+06,511.000000,2407.000000,25717.000000,5300.000000
50%,7.505302e+06,1078.000000,3665.000000,46054.000000,5499.000000
75%,7.520247e+06,1535.000000,4938.000000,67374.000000,5812.000000
max,7.540041e+06,6956.000000,10794.000000,105252.000000,9402.000000


In [125]:
users_df.describe()

,id,current_age,retirement_age,birth_year,birth_month,latitude,longitude,credit_score,num_credit_cards
count,2600.000000,2537.000000,2535.000000,2537.000000,2545.000000,2550.000000,2546.000000,2551.000000,2538.000000
mean,1043.575769,45.400473,66.250493,1973.759559,6.436149,37.413635,-91.480157,709.828303,3.059890
std,703.990346,18.420508,3.612874,18.427102,3.557252,5.136674,16.264821,66.770226,1.642807
min,0.000000,18.000000,50.000000,1918.000000,1.000000,20.880000,-159.410000,480.000000,1.000000
25%,511.000000,30.000000,65.000000,1961.000000,3.000000,33.860000,-97.340000,681.000000,2.000000
50%,1011.000000,44.000000,66.000000,1975.000000,7.000000,38.280000,-86.335000,711.000000,3.000000
75%,1515.250000,58.000000,68.000000,1989.000000,10.000000,41.290000,-80.072500,753.000000,4.000000
max,6840.000000,101.000000,79.000000,2002.000000,12.000000,61.200000,-68.670000,850.000000,9.000000


In [126]:
cards_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: double (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



In [127]:
transactions_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



In [128]:
users_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



## Identificar valores faltantes

In [129]:
cards_df.isnull().sum()

id                        1
client_id                 1
card_brand               72
card_type                57
card_number              63
expires                  66
cvv                      68
has_chip                 68
num_cards_issued         62
credit_limit             91
acct_open_date           55
year_pin_last_changed    72
card_on_dark_web         76
dtype: int64

In [130]:
transactions_df.isnull().sum()

id                    0
date                 99
client_id             2
card_id               2
amount               99
use_chip             81
merchant_id           3
merchant_city        99
merchant_state     5559
zip                5742
mcc                 100
errors            49300
dtype: int64

In [131]:
users_df.isnull().sum()

id                    0
current_age          63
retirement_age       65
birth_year           63
birth_month          55
gender               61
address              68
latitude             50
longitude            54
per_capita_income    64
yearly_income        46
total_debt           63
credit_score         49
num_credit_cards     62
dtype: int64

In [132]:
cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
]).show()

+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
| id|client_id|card_brand|card_type|card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  1|        1|        72|       57|         63|     66| 68|      68|              62|          91|            55|                   72|              76|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+



In [133]:
transactions_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_spark.columns
]).show()

+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+
| id|date|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state| zip|mcc|errors|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+
|  0|  99|        2|      2|    99|      81|          3|           99|          5559|5742|100| 49300|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+----+---+------+



In [134]:
users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
]).show()

+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|         63|            65|        63|         55|    61|     68|      50|       54|               64|           46|        63|          49|              62|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+



## Imputación de datos con Pandas

In [135]:
# Imputar la edad actual utilizando la mediana

mediana = users_df["current_age"].median()

users_df["current_age"] = users_df["current_age"].fillna(mediana)

In [136]:
# Imputar el género utilizando la moda

moda = users_df["gender"].mode()[0]

users_df["gender"] = users_df["gender"].fillna(moda)

## Imputación de datos con Apache Spark

In [137]:
imputer = Imputer(
    inputCols=["current_age"],
    outputCols=["current_age"]
)

users_spark = imputer.fit(users_spark).transform(users_spark)

## Verificar los cambios

In [138]:
users_df.isnull().sum()

id                    0
current_age           0
retirement_age       65
birth_year           63
birth_month          55
gender                0
address              68
latitude             50
longitude            54
per_capita_income    64
yearly_income        46
total_debt           63
credit_score         49
num_credit_cards     62
dtype: int64